# Assemblying RPE1 Dataset
You will find here all the code required to get the data used in the general tutorial ``1_general_tutorial.ipynb``, the data are from [Statistical inference with a manifold-constrained RNA velocity model uncovers cell cycle speed modulations](https://www.nature.com/articles/s41592-024-02471-8).

Running time depends on your internet connection but you could expect it to run in the order of a few minutes.

In [1]:
DATA_FOLDER = "./data/"  # path to the data folder.

## Package and Environment Set-Up
Not all these steps are required to run this notebook, but they are required to run CoPhaser:

0) Download this repo
1) Create a [conda environment](https://www.anaconda.com/docs/getting-started/miniconda/main)
```bash
conda create -n "CoPhaser_Env" python=3.13
conda activate CoPhaser_Env
```
2) Install PyTorch with CUDA support (if you want to use the GPU)
   **Note:** Check your CUDA driver version (`nvidia-smi`) and install the compatible PyTorch version. 
   Go to [the official page](https://pytorch.org/) for more information. The code was tested using the version 2.9.1.
   
   To dowload CUDA 12.6, for instance, with the latest PyTorch version:
```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
```
3) Install the package
```bash
cd CoPhaser/
pip install .
```

## Dowload the files
Dowload the counts data (363 mb)

In [2]:
import requests
from tqdm import tqdm
import os

# create the data folder if it doesn't exist
os.makedirs(DATA_FOLDER, exist_ok=True)


def dowload_file(url, output_path):
    response = requests.get(url, stream=True)

    if response.status_code == 200:
        total_size = int(response.headers.get("content-length", 0))
        chunk_size = 1024 * 1024  # 1 MB

        with open(output_path, "wb") as f, tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

        print("Download completed successfully.")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")


output_paths = []
for rep in [1, 2]:
    print("Dowloading the 2 files...")
    file_name = f"RPE_37C_ctrl_Rep{rep}.h5ad.gz"
    output_path = DATA_FOLDER + "/" + file_name
    dowload_file(
        f"https://zenodo.org/records/12517650/files/{file_name}?download=1", output_path
    )
    output_paths.append(output_path)

Dowloading the 2 files...


100%|██████████| 100M/100M [00:07<00:00, 14.1MB/s] 


Download completed successfully.
Dowloading the 2 files...


100%|██████████| 263M/263M [00:06<00:00, 43.0MB/s] 

Download completed successfully.


## Open the files

In [3]:
import gzip
import shutil
import scanpy as sc

adatas = []
for path in output_paths:
    with gzip.open(path, "rb") as f_in:
        output = path[:-3]
        with open(output, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    adatas.append(sc.read_h5ad(output))

## Concatenate and save the RPE datasets

In [4]:
import os

adata = sc.concat(adatas)
# save the datasets
adata.write_h5ad(os.path.dirname(output_paths[0]) + "/rpe1_velocycle.h5ad")